# 01a_Metadaten_Intake

Dieses Notebook lädt **nur** die zwei JSON-Quellen und erzeugt eine roh harmonisierte Tabelle.

- `tiktok_100_vs_100/Top_100/top_metadata.json`
- `tiktok_100_vs_100/Normal_100/normal_metadata.json`

**Artefakte (neben diesem Notebook):** `metadata_raw.parquet`, `metadata_provenance.csv`

## Governance & Guardrails
- Quelle fixiert (nur 2 JSONs)
- Zeiten in UTC
- Provenienz (source_file, group)
- deterministisch

## 1) Setup & Pfade

In [ ]:

from pathlib import Path
import pandas as pd, numpy as np, json, re, warnings
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path("..").resolve()
TIKTOK_DIR = PROJECT_ROOT / "tiktok_100_vs_100"

TOP_JSON = TIKTOK_DIR / "Top_100" / "top_metadata.json"
NORMAL_JSON = TIKTOK_DIR / "Normal_100" / "normal_metadata.json"

FALLBACK_TOP = Path("/mnt/data/top_metadata.json")
FALLBACK_NORMAL = Path("/mnt/data/normal_metadata.json")
if not TOP_JSON.exists() and FALLBACK_TOP.exists():
    TOP_JSON = FALLBACK_TOP
if not NORMAL_JSON.exists() and FALLBACK_NORMAL.exists():
    NORMAL_JSON = FALLBACK_NORMAL

print("TOP_JSON:", TOP_JSON, TOP_JSON.exists())
print("NORMAL_JSON:", NORMAL_JSON, NORMAL_JSON.exists())


## 2) Helper

In [ ]:

def snake(s: str): 
    import re
    return re.sub(r"\W+", "_", str(s).strip()).strip("_").lower()

def cols_to_snake(df):
    df = df.copy()
    df.columns = [snake(c) for c in df.columns]
    return df

def read_json_table(path: Path, group_name: str):
    import json, pandas as pd
    assert path.exists(), f"Datei fehlt: {path}"
    try:
        data = json.loads(path.read_text(encoding="utf-8"))
        if isinstance(data, dict) and "data" in data and isinstance(data["data"], list):
            df = pd.DataFrame(data["data"])
        else:
            df = pd.DataFrame(data if isinstance(data, list) else [data])
    except Exception:
        df = pd.read_json(path, lines=True)
    df = cols_to_snake(df)
    df["source_file"] = str(path)
    df["group"] = group_name
    return df

def unify_types(df):
    df = df.copy()
    for c in ["upload_time","create_time","timestamp","create_date"]:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], utc=True, errors="coerce")
    for c in ["duration_s","duration","creator_follower_count","creator_posts_count",
              "views","view_count","likes","like_count","comments","comment_count","shares","share_count","rank"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    for c in ["creator_verified","verified"]:
        if c in df.columns:
            df[c] = df[c].fillna(False).astype(bool)
    rename = {
        "verified":"creator_verified",
        "like_count":"likes", "view_count":"views", "comment_count":"comments", "share_count":"shares",
        "duration":"duration_s",
        "id":"video_id", "aweme_id":"video_id", "tiktok_id":"video_id"
    }
    for k,v in rename.items():
        if k in df.columns and v not in df.columns:
            df = df.rename(columns={k:v})
    if "video_id" in df.columns:
        df["video_id"] = df["video_id"].astype(str)
    return df

def pick_one_row_per_video(df):
    time_col = next((c for c in ["upload_time","create_time","timestamp","create_date"] if c in df.columns), None)
    if time_col:
        df = df.sort_values(["video_id", time_col], ascending=[True, False])
    else:
        df = df.sort_values(["video_id"])
    return df.drop_duplicates("video_id", keep="first").reset_index(drop=True)


## 3) Quellen laden

In [ ]:

sources = []
if TOP_JSON.exists():
    sources.append(("top", TOP_JSON))
if NORMAL_JSON.exists():
    sources.append(("normal", NORMAL_JSON))

print("Gefundene Dateien:", len(sources), [str(p) for _, p in sources])
assert len(sources) == 2, "Es sollten genau zwei JSON-Dateien gefunden werden."

parts = []
for grp, path in sources:
    dfp = read_json_table(path, grp)
    dfp = unify_types(dfp)
    parts.append(dfp)

meta_raw = pd.concat(parts, ignore_index=True, sort=False)
meta_raw.shape, meta_raw.head(3)


## 4) Harmonisierung & eine Zeile pro video_id

In [ ]:

assert "video_id" in meta_raw.columns, "Spalte 'video_id' fehlt in den JSONs."
before = meta_raw.shape[0]
meta_one = pick_one_row_per_video(meta_raw)
after = meta_one.shape[0]
print(f"Dedupliziert: {before} → {after}")
meta_one.head(5)


## 5) (Optional) Proxy-Label (nur QC)

In [ ]:

proxy_map = {"top":1, "normal":0}
if "group" in meta_one.columns and "is_viral_proxy" not in meta_one.columns:
    meta_one["is_viral_proxy"] = meta_one["group"].map(proxy_map).fillna(0).astype(int)
meta_one[["video_id","group","is_viral_proxy"]].head(5)


## 6) Artefakte speichern

In [ ]:

raw_out = Path("metadata_raw.parquet")
meta_one.to_parquet(raw_out, index=False)
print("Gespeichert:", raw_out.resolve())

prov = meta_one[["video_id","source_file","group"]].drop_duplicates("video_id")
prov_out = Path("metadata_provenance.csv")
prov.to_csv(prov_out, index=False)
print("Provenienz gespeichert:", prov_out.resolve())


---

### Weiter mit 01b_Metadaten_Cleaning_QC
In 01b validieren wir Schema, prüfen Missingness & Leakage und erzeugen `metadata_base.parquet`.